In [ ]:
import os
import sys
import time
import torch
import contextlib
import json
from typing import TypedDict

import wave
from datasets import load_dataset
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    AutomaticSpeechRecognitionPipeline,
    pipeline,
)


def get_wav_duration_seconds(wav_path: str) -> float:
    """Retourne la durée d'un fichier WAV en secondes."""
    with contextlib.closing(wave.open(wav_path, "r")) as f:
        frames = f.getnframes()
        rate = f.getframerate()
        if rate > 0:
            return frames / float(rate)
    return -1


def adjust_pauses_for_hf_pipeline_output(pipeline_output, split_threshold=0.12):
    """un
    Adjust pause timings by distributing pauses up to the threshold evenly between adjacent words.
    """

    adjusted_chunks = pipeline_output["chunks"].copy()

    for i in range(len(adjusted_chunks) - 1):
        current_chunk = adjusted_chunks[i]
        next_chunk = adjusted_chunks[i + 1]

        current_start, current_end = current_chunk["timestamp"]
        next_start, next_end = next_chunk["timestamp"]
        pause_duration = next_start - current_end

        if pause_duration > 0:
            if pause_duration > split_threshold:
                distribute = split_threshold / 2
            else:
                distribute = pause_duration / 2

            # Adjust current chunk end time
            adjusted_chunks[i]["timestamp"] = (current_start, current_end + distribute)

            # Adjust next chunk start time
            adjusted_chunks[i + 1]["timestamp"] = (next_start - distribute, next_end)
    pipeline_output["chunks"] = adjusted_chunks

    return pipeline_output


def build_model() -> AutomaticSpeechRecognitionPipeline:
    # Resolve torch backend (CPU vs GPU)
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    # Load the model and processor
    model_id = "nyrahealth/CrisperWhisper"
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
    )
    model.to(device)
    processor = AutoProcessor.from_pretrained(model_id)

    # Prepare the pipeline
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        chunk_length_s=30,
        batch_size=16,
        return_timestamps="word",
        torch_dtype=torch_dtype,
        device=device,
    )
    return pipe


def process_file(file: str, pipe: AutomaticSpeechRecognitionPipeline) -> dict:
    result = pipe(file)
    adjusted_result = adjust_pauses_for_hf_pipeline_output(result)
    return adjusted_result  # type: ignore


pipe = build_model()
res = process_file("./data/sample/begaiement.wav", pipe)
res

In [ ]:
class TranscriptionResult(TypedDict):
    audio_file: str
    torch_backend: str
    torch_dtype: str
    audio_duration_s: float
    pipeline_creation_time_s: float
    transcription_time_s: float


def process_folder(
    input_folder: str, output_folder: str, overwrite_existing: bool = False
):

    # Load the model
    t0 = time.time()
    pipe = build_model()
    pipeline_creation_time = time.time() - t0

    # Process each file in the folder
    for filename in os.listdir(input_folder):
        if not filename.endswith(".wav"):
            continue

        input_file = f"{input_folder}/{filename}"
        output_file = f"{output_folder}/output/{filename[:-4]}.json"
        metadata_file = f"{output_folder}/metadata/{filename[:-4]}_metadata.json"

        if (
            os.path.exists(output_file)
            and os.path.exists(metadata_file)
            and not overwrite_existing
        ):
            print(f"Skipping {filename} (already processed)")
            continue

        print(f"> Processing {filename}...")
        t0 = time.time()
        result = process_file(input_file, pipe)
        transcription_time = time.time() - t0

        # Save result
        with open(output_file, "w") as f:
            json.dump(result, f, indent=2)

        # Save metadata
        metadata = TranscriptionResult(
            audio_file=input_file,
            torch_backend=str(pipe.device),
            torch_dtype=str(pipe.torch_dtype),
            audio_duration_s=get_wav_duration_seconds(input_file),
            pipeline_creation_time_s=pipeline_creation_time,
            transcription_time_s=transcription_time,
        )
        with open(metadata_file, "w") as f:
            json.dump(metadata, f, indent=2)
        print(f'\t> Audio duration:\t{metadata["audio_duration_s"]:.2f}s')
        print(f"\t> Transcribed in:\t{transcription_time:.2f}s")


process_folder("./data/input", "./data/crisper-whisper", overwrite_existing=False)